In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
## general libraries
import pathlib
from rich.pretty import install, pprint

## data handling libraries
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.dask import TqdmCallback as ProgressBarDask

# plotting libraries
import matplotlib.pyplot as plt

## machine learning libraries
import gpytorch 
import torch

## PAMIR libraries
import mlflow
import pamir_mlpermafrost as pamir
from cryogrid_pytools import xr_raster_vector
import cryogrid_pytools as cg
import plots

install(overflow=True)

In [ ]:
def process_col_name(s):
    seas, depth = s.strip('ground_temp').split('_')
    depth = float('0.' + depth[1:]) if depth.startswith('0') else float(depth)
    return seas, depth

In [ ]:
exp = 'N180_exp4'
store_exp = f'simplecache::s3://spi-pamir-cryogrid/processed-cluster_config/cluster_config-k1500-pamir_{exp.replace("_", "-")}.zarr/'
ds_exp = xr.open_zarr(store_exp, storage_options=pamir.data.s3_utils.get_fsspec_kwargs()).assign_coords(exp=[exp], tag=lambda x: x.tag % 10_000)
da_clusters = ds_exp.cluster_labels.compute()


store_spatial = f"simplecache::s3://spi-pamir-cryogrid/pamir-MLpermafrost/data-inference/inference_variables-710w365s750e400n-100m.zarr/"
ds_spatial = xr.open_zarr(store_spatial, storage_options=pamir.data.s3_utils.get_fsspec_kwargs())

df_train = pd.read_parquet('../../data/training/training_data-k1500-pamir_ns180-expX-ground_temp_seas_depth.parquet')
targets = df_train.filter(regex='ground_temp_*')
targets.columns = pd.MultiIndex.from_tuples([process_col_name(s) for s in targets.columns], names=['season', 'depth'])

snow = (ds_spatial.land_cover == 9).compute().morph.clean().astype(int).interp_like(da_clusters, method='nearest').astype(bool)

In [ ]:
da_target = targets.stack(level=[0, 1]).to_xarray()

In [ ]:
da = da_target.sel(exp=ds_exp.exp.item(), drop=True).rename(tag='index').stack(season_depth=['season', 'depth'])

In [ ]:
out = []
for season_depth in da.season_depth.values:
    print(season_depth)
    da_sd = da.sel(season_depth=season_depth)
    da_mapped = cg.spatial_clusters.map_gridcells_to_clusters(da_sd, da_clusters)
    out.append(da_mapped)
ground_temp = (
    xr.concat(out, dim='season_depth')
    .set_index(season_depth=['season', 'depth'])
    .unstack('season_depth')
    .transpose('season', 'depth', ...)
    .chunk({'season': 1, 'depth': 1, 'x': 1111, 'y': 1111})
    .persist())

In [ ]:
for season in ['JFM', 'JAS']:
    fig, axs, cbar = plots.plot_depths(
        ground_temp.sel(season=season).coarsen(x=5, y=5, boundary='pad').mean(),
        title=f'Ground Temperature for $\\mathbf{{\\overline{{{season}}}}}$ [2000-2024]',
        info='Point locations modelled with CryoGrid clustering approach mapped to clusters',
        vmin=-10, vmax=10, cmap='RdBu_r',
        path_to_zarr=store_exp)
    cbar.set_label('Ground Temperature [°C]')
    fig.savefig(f'./figs/cg-{exp}-ground_temp_{season}.png', dpi=300, bbox_inches='tight', transparent=True, facecolor='none')

    plt.close('all')

In [ ]:
ground_temp_diff = (ground_temp[0] - ground_temp[1]).coarsen(x=3, y=3, boundary='pad').mean().persist()

In [ ]:
fig, axs, cbar = plots.plot_depths(
    ground_temp_diff,
    path_to_zarr=store_exp,
    title='Ground Temperature Difference (JAS - JFM)',
    info='Point locations modelled with CryoGrid clustering approach mapped to clusters',
    vmin=-2, vmax=17, cmap='Spectral_r'
)

cbar.set_label('JAS $-$ JFM Ground Temperature difference [°C]')

fig.savefig(f'./figs/cg-{exp}-ground_temp_diff_JAS_minus_JFM.png', dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
plt.close(fig)

# Permafrost

In [ ]:
summer = ground_temp.sel(season='JAS')
winter = ground_temp.sel(season='JFM')

summer_interp = summer.interp(depth=np.arange(0.25, 7.25, 0.125), method='linear')

summer_frozen = summer < 0
summer_interp_frozen = summer_interp < 0
winter_frozen = winter < 0

mask = summer.isnull().all('depth')

permafrost = (summer_frozen & winter_frozen).any(dim='depth')

In [ ]:
with ProgressBarDask():
    da = (
        summer_interp_frozen
        .idxmax(dim='depth')
        .persist()
        .where(~snow)
        .fillna(0.20)
        .where(permafrost | snow)
        .drop_vars('season')
        # .rolling(x=3, y=3, center=True, min_periods=1).median()
        # .rolling(x=3, y=3, center=True, min_periods=1).median()
        .compute()
    )

In [ ]:
cmap = plt.cm.GnBu_r
cmap.set_under("#7c2f5f")
cmap.set_bad('none')

fig, axs = plt.subplots(figsize=[8, 6], dpi=300)
img = da.plot.imshow(
    vmin=0.24, 
    vmax=4, 
    cmap=cmap, 
    ax=axs, 
    cbar_kwargs=dict(extendrect=True, extendfrac=0.08, pad=0.02))

cbar = img.colorbar
cbar.ax.axhline(0.24, color='w', lw=4, zorder=10)
cbar.set_ticks([0.24, 1, 2, 3, 4])
cbar.set_ticklabels(['\nPermanent\nSnow', '1', '2', '3', '4'])
cbar.set_label('Depth of permafrost [m]', rotation=90, labelpad=-35, size=10)

axs.set_title('Depth of permafrost based on summer ground temperatures', loc='left')
axs.set_xlabel('')

fig.subplots_adjust(hspace=0.08)

plots.add_plot_meta(
    axs, 
    info='CryoGrid point locations mapped to cluster labels at 100 m.\n         Reported depth is the freeze-thaw boundary depth.',
    path=store_exp, 
    size=6,
)

fig.savefig(f'./figs/cg-{exp}-permafrost_depth.png', dpi=300, bbox_inches='tight', transparent=True, facecolor='none')